# NB_bishop_ch11_figures

**Bishop Chapter 11 — Key Figures: Structured Distributions / Graphical Models**

This notebook generates pedagogical matplotlib diagrams for Bishop Chapter 11 on graphical models: Bayesian networks, conditional independence, Markov blankets, Markov random fields, factor graphs, and belief propagation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_11/NB_bishop_ch11_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle, Rectangle
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

# ── helper functions for graphical model diagrams ───────────────
def draw_node(ax, xy, label, radius=0.3, fc='white', ec='black', lw=2,
              fontsize=14, fontcolor='black', zorder=10):
    """Draw a circular node with a centered label."""
    circle = plt.Circle(xy, radius, fc=fc, ec=ec, lw=lw, zorder=zorder)
    ax.add_patch(circle)
    ax.text(xy[0], xy[1], label, fontsize=fontsize, ha='center', va='center',
            fontweight='bold', color=fontcolor, zorder=zorder + 1)
    return circle

def draw_square_node(ax, xy, label, size=0.4, fc='black', ec='black', lw=2,
                     fontsize=12, fontcolor='white', zorder=10):
    """Draw a square factor node with a centered label."""
    rect = Rectangle((xy[0] - size/2, xy[1] - size/2), size, size,
                      fc=fc, ec=ec, lw=lw, zorder=zorder)
    ax.add_patch(rect)
    ax.text(xy[0], xy[1], label, fontsize=fontsize, ha='center', va='center',
            fontweight='bold', color=fontcolor, zorder=zorder + 1)
    return rect

def draw_directed_edge(ax, start, end, node_radius=0.3, color='black', lw=1.8):
    """Draw a directed arrow between two nodes, stopping at the node boundary."""
    dx = end[0] - start[0]
    dy = end[1] - start[1]
    dist = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / dist, dy / dist
    x0 = start[0] + ux * node_radius
    y0 = start[1] + uy * node_radius
    x1 = end[0] - ux * node_radius
    y1 = end[1] - uy * node_radius
    arrow = FancyArrowPatch((x0, y0), (x1, y1),
                            arrowstyle='->', mutation_scale=18,
                            color=color, lw=lw, zorder=5)
    ax.add_patch(arrow)
    return arrow

def draw_undirected_edge(ax, start, end, node_radius=0.3, color='black', lw=1.8):
    """Draw an undirected edge between two nodes."""
    dx = end[0] - start[0]
    dy = end[1] - start[1]
    dist = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / dist, dy / dist
    x0 = start[0] + ux * node_radius
    y0 = start[1] + uy * node_radius
    x1 = end[0] - ux * node_radius
    y1 = end[1] - uy * node_radius
    ax.plot([x0, x1], [y0, y1], color=color, lw=lw, zorder=5)

print('Setup complete.')

## Figure 11.1 — Bayesian Network (Directed Acyclic Graph)

A simple DAG with 5 nodes and directed edges: $A \to B \to D$, $A \to C \to D$, $C \to E$.

In [ ]:
# ── Figure 11.1: Bayesian Network DAG ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.axis('off')

# Node positions (top-down layout)
nodes = {
    'A': (2, 4),
    'B': (0.5, 2.5),
    'C': (3.5, 2.5),
    'D': (2, 1),
    'E': (4.5, 1),
}

# Directed edges
edges = [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('C', 'E')]

# Draw edges first (behind nodes)
for src, dst in edges:
    draw_directed_edge(ax, nodes[src], nodes[dst],
                       node_radius=0.3, color=COLORS['blue'], lw=2)

# Draw nodes
node_colors = {
    'A': '#e8edf3',
    'B': '#fff3e0',
    'C': '#fff3e0',
    'D': '#e8f5e9',
    'E': '#e8f5e9',
}
for name, pos in nodes.items():
    draw_node(ax, pos, name, radius=0.3, fc=node_colors[name],
              ec=COLORS['blue'], fontsize=16, fontcolor=COLORS['blue'])

# Joint distribution annotation
ax.text(2, -0.5,
        r'$p(A,B,C,D,E) = p(A)\,p(B|A)\,p(C|A)\,p(D|B,C)\,p(E|C)$',
        fontsize=12, ha='center', va='center', color=COLORS['blue'])

ax.set_title('Bayesian Network (Directed Graphical Model)', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig11_1_bayesian_network')
plt.show()

## Figure 11.3 — Conditional Independence and D-Separation

Three canonical structures: (a) head-to-tail (chain), (b) tail-to-tail (fork), (c) head-to-head (collider / v-structure).

In [ ]:
# ── Figure 11.3: D-Separation (three canonical structures) ─────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

r = 0.3  # node radius

# ── (a) Head-to-tail (chain): A → B → C ───────────────────────
ax = axes[0]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-1.5, 2.0)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('(a) Head-to-Tail (Chain)', fontsize=12, pad=10)

pos_a = {'A': (0, 0.5), 'B': (2, 0.5), 'C': (4, 0.5)}
for src, dst in [('A', 'B'), ('B', 'C')]:
    draw_directed_edge(ax, pos_a[src], pos_a[dst], r, COLORS['blue'], 2)

draw_node(ax, pos_a['A'], '$a$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])
draw_node(ax, pos_a['B'], '$b$', r, '#fce4ec', COLORS['red'], fontcolor=COLORS['red'])
draw_node(ax, pos_a['C'], '$c$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])

# Observed node indicator (shaded)
ax.text(2, -0.3, 'observed', fontsize=9, ha='center', color=COLORS['red'], style='italic')
ax.text(2, -0.7, r'$a \perp\!\!\!\perp c \mid b$', fontsize=13, ha='center',
        color=COLORS['blue'])

# ── (b) Tail-to-tail (fork): A ← B → C ───────────────────────
ax = axes[1]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-1.5, 2.0)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('(b) Tail-to-Tail (Fork)', fontsize=12, pad=10)

pos_b = {'A': (0, 0.5), 'B': (2, 0.5), 'C': (4, 0.5)}
draw_directed_edge(ax, pos_b['B'], pos_b['A'], r, COLORS['blue'], 2)
draw_directed_edge(ax, pos_b['B'], pos_b['C'], r, COLORS['blue'], 2)

draw_node(ax, pos_b['A'], '$a$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])
draw_node(ax, pos_b['B'], '$b$', r, '#fce4ec', COLORS['red'], fontcolor=COLORS['red'])
draw_node(ax, pos_b['C'], '$c$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])

ax.text(2, -0.3, 'observed', fontsize=9, ha='center', color=COLORS['red'], style='italic')
ax.text(2, -0.7, r'$a \perp\!\!\!\perp c \mid b$', fontsize=13, ha='center',
        color=COLORS['blue'])

# ── (c) Head-to-head (collider): A → B ← C ───────────────────
ax = axes[2]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-1.5, 2.0)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('(c) Head-to-Head (Collider)', fontsize=12, pad=10)

pos_c = {'A': (0, 0.5), 'B': (2, 0.5), 'C': (4, 0.5)}
draw_directed_edge(ax, pos_c['A'], pos_c['B'], r, COLORS['blue'], 2)
draw_directed_edge(ax, pos_c['C'], pos_c['B'], r, COLORS['blue'], 2)

draw_node(ax, pos_c['A'], '$a$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])
draw_node(ax, pos_c['B'], '$b$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])
draw_node(ax, pos_c['C'], '$c$', r, '#e8edf3', COLORS['blue'], fontcolor=COLORS['blue'])

ax.text(2, -0.3, 'not observed', fontsize=9, ha='center', color=COLORS['green'], style='italic')
ax.text(2, -0.7, r'$a \perp\!\!\!\perp c$  (marginally)', fontsize=13, ha='center',
        color=COLORS['blue'])
ax.text(2, -1.15, r'but $a \not\!\perp\!\!\!\perp c \mid b$', fontsize=11, ha='center',
        color=COLORS['red'])

fig.tight_layout()
save_fig(fig, 'fig11_3_conditional_independence')
plt.show()

## Figure 11.5 — Markov Blanket

The Markov blanket of node $x_i$ consists of its parents, children, and co-parents (other parents of its children). Given the blanket, $x_i$ is conditionally independent of all other nodes.

In [ ]:
# ── Figure 11.5: Markov Blanket ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(-2, 8)
ax.set_ylim(-2, 8)
ax.set_aspect('equal')
ax.axis('off')

r = 0.35

# Central node
xi_pos = (3, 3.5)

# Parents of xi
parents = {'P1': (1.5, 5.5), 'P2': (4.5, 5.5)}

# Children of xi
children = {'C1': (1.5, 1.5), 'C2': (4.5, 1.5)}

# Co-parents (other parents of children, not xi)
coparents = {'CP1': (0, 3.0), 'CP2': (6, 3.0)}

# Other nodes (outside blanket)
others = {'O1': (1.5, 7.5), 'O2': (4.5, 7.5), 'O3': (0, 0), 'O4': (6, 0)}

# Draw Markov blanket shaded region
from matplotlib.patches import Ellipse
blanket = Ellipse(xy=(3, 3.5), width=8.5, height=6.5, angle=0,
                  fc=COLORS['amber'], alpha=0.10, ec=COLORS['amber'],
                  lw=2.5, ls='--', zorder=1)
ax.add_patch(blanket)
ax.text(7.0, 6.2, 'Markov\nblanket', fontsize=12, color=COLORS['amber'],
        ha='center', style='italic', fontweight='bold')

# ── Edges ──────────────────────────────────────────────────────
# Parents → xi
for p in parents.values():
    draw_directed_edge(ax, p, xi_pos, r, 'gray', 1.8)

# xi → children
for c in children.values():
    draw_directed_edge(ax, xi_pos, c, r, 'gray', 1.8)

# Co-parents → children
draw_directed_edge(ax, coparents['CP1'], children['C1'], r, 'gray', 1.8)
draw_directed_edge(ax, coparents['CP2'], children['C2'], r, 'gray', 1.8)

# Other → parents (to show wider graph)
draw_directed_edge(ax, others['O1'], parents['P1'], r, 'lightgray', 1.2)
draw_directed_edge(ax, others['O2'], parents['P2'], r, 'lightgray', 1.2)

# Children → others
draw_directed_edge(ax, children['C1'], others['O3'], r, 'lightgray', 1.2)
draw_directed_edge(ax, children['C2'], others['O4'], r, 'lightgray', 1.2)

# ── Draw nodes ─────────────────────────────────────────────────
# Central node (target)
draw_node(ax, xi_pos, '$x_i$', r, '#fce4ec', COLORS['red'], fontsize=16,
          fontcolor=COLORS['red'])

# Parents
for name, pos in parents.items():
    draw_node(ax, pos, name.replace('P', '$p_') + '$', r, '#e8edf3',
              COLORS['blue'], fontcolor=COLORS['blue'])

# Children
for name, pos in children.items():
    draw_node(ax, pos, name.replace('C', '$c_') + '$', r, '#e8f5e9',
              COLORS['green'], fontcolor=COLORS['green'])

# Co-parents
for name, pos in coparents.items():
    draw_node(ax, pos, name.replace('CP', '$u_') + '$', r, '#fff3e0',
              COLORS['amber'], fontcolor=COLORS['amber'])

# Others (outside blanket, faded)
for name, pos in others.items():
    draw_node(ax, pos, '', r, '#f5f5f5', 'lightgray')

# ── Legend annotations ─────────────────────────────────────────
legend_x, legend_y = -1.5, 7.5
labels = [
    (COLORS['red'], '$x_i$: target node'),
    (COLORS['blue'], 'Parents'),
    (COLORS['green'], 'Children'),
    (COLORS['amber'], 'Co-parents'),
]
for i, (c, lab) in enumerate(labels):
    ax.plot(legend_x, legend_y - i * 0.6, 'o', ms=10, color=c, zorder=10)
    ax.text(legend_x + 0.3, legend_y - i * 0.6, lab, fontsize=10, va='center', color=c)

ax.set_title('Markov Blanket', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig11_5_markov_blanket')
plt.show()

## Figure 11.7 — Undirected Graphical Model (Markov Random Field)

A 2x2 grid MRF with maximal cliques shaded. In undirected models the joint distribution factorizes over clique potentials.

In [ ]:
# ── Figure 11.7: Undirected Graphical Model (MRF) ─────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(-1.5, 7.5)
ax.set_ylim(-2.0, 6.0)
ax.set_aspect('equal')
ax.axis('off')

r = 0.35
sp = 3.0  # spacing

# 2x3 grid of nodes
grid_pos = {}
labels_grid = [['$x_1$', '$x_2$', '$x_3$'],
               ['$x_4$', '$x_5$', '$x_6$']]
for row in range(2):
    for col in range(3):
        name = f'x{row * 3 + col + 1}'
        grid_pos[name] = (col * sp, (1 - row) * sp)

# Shade cliques (each pair of connected nodes forms a clique)
# Highlight two example maximal cliques with shading
from matplotlib.patches import Polygon

# Clique {x1, x2, x4, x5} — forms a square
clique1 = Polygon([
    (grid_pos['x1'][0] - 0.5, grid_pos['x1'][1] + 0.5),
    (grid_pos['x2'][0] + 0.5, grid_pos['x2'][1] + 0.5),
    (grid_pos['x5'][0] + 0.5, grid_pos['x5'][1] - 0.5),
    (grid_pos['x4'][0] - 0.5, grid_pos['x4'][1] - 0.5),
], closed=True, fc=COLORS['blue'], alpha=0.08, ec=COLORS['blue'],
   lw=2, ls='--', zorder=1)
ax.add_patch(clique1)
ax.text(-0.3, 1.5, r'$\psi_1$', fontsize=14, color=COLORS['blue'],
        ha='center', fontweight='bold')

# Clique {x2, x3, x5, x6}
clique2 = Polygon([
    (grid_pos['x2'][0] - 0.5, grid_pos['x2'][1] + 0.5),
    (grid_pos['x3'][0] + 0.5, grid_pos['x3'][1] + 0.5),
    (grid_pos['x6'][0] + 0.5, grid_pos['x6'][1] - 0.5),
    (grid_pos['x5'][0] - 0.5, grid_pos['x5'][1] - 0.5),
], closed=True, fc=COLORS['green'], alpha=0.08, ec=COLORS['green'],
   lw=2, ls='--', zorder=1)
ax.add_patch(clique2)
ax.text(6.3, 1.5, r'$\psi_2$', fontsize=14, color=COLORS['green'],
        ha='center', fontweight='bold')

# Draw undirected edges (grid connectivity)
edges_u = [
    ('x1', 'x2'), ('x2', 'x3'),       # top row
    ('x4', 'x5'), ('x5', 'x6'),       # bottom row
    ('x1', 'x4'), ('x2', 'x5'), ('x3', 'x6'),  # vertical
]
for a, b in edges_u:
    draw_undirected_edge(ax, grid_pos[a], grid_pos[b], r, COLORS['blue'], 2)

# Draw nodes
for name, pos in grid_pos.items():
    idx = name.replace('x', '')
    draw_node(ax, pos, f'$x_{idx}$', r, 'white', COLORS['blue'],
              fontsize=14, fontcolor=COLORS['blue'])

# Factorization annotation
ax.text(3, -1.5,
        r'$p(\mathbf{x}) = \frac{1}{Z}\,\prod_{C} \psi_C(\mathbf{x}_C)$'
        r'  where $Z = \sum_{\mathbf{x}} \prod_C \psi_C(\mathbf{x}_C)$',
        fontsize=13, ha='center', va='center', color=COLORS['blue'])

ax.set_title('Undirected Graphical Model (Markov Random Field)', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig11_7_undirected_graph')
plt.show()

## Figure 11.9 — Factor Graph

Factor graphs make the factorization explicit by introducing factor nodes (squares) connected to variable nodes (circles). Every edge connects a factor to a variable.

In [ ]:
# ── Figure 11.9: Factor Graph ──────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(-1, 13)
ax.set_ylim(-2, 4)
ax.set_aspect('equal')
ax.axis('off')

r = 0.35       # variable node radius
sq = 0.45      # factor node size

# Variable nodes (circles) — evenly spaced
var_nodes = {
    '$x_1$': (0, 1),
    '$x_2$': (3, 1),
    '$x_3$': (6, 1),
    '$x_4$': (9, 1),
    '$x_5$': (12, 1),
}

# Factor nodes (squares) — between variables and connecting to them
factor_nodes = {
    '$f_a$': (1.5, 1),    # connects x1, x2
    '$f_b$': (4.5, 1),    # connects x2, x3
    '$f_c$': (7.5, 1),    # connects x3, x4
    '$f_d$': (10.5, 1),   # connects x4, x5
    '$f_e$': (3, 3),      # connects x2 only (prior / unary factor)
    '$f_f$': (9, 3),      # connects x4 only
}

# Edges: each factor connects to its variables
factor_edges = {
    '$f_a$': ['$x_1$', '$x_2$'],
    '$f_b$': ['$x_2$', '$x_3$'],
    '$f_c$': ['$x_3$', '$x_4$'],
    '$f_d$': ['$x_4$', '$x_5$'],
    '$f_e$': ['$x_2$'],
    '$f_f$': ['$x_4$'],
}

# Draw edges (behind everything)
for fname, connected_vars in factor_edges.items():
    fpos = factor_nodes[fname]
    for vname in connected_vars:
        vpos = var_nodes[vname]
        dx = vpos[0] - fpos[0]
        dy = vpos[1] - fpos[1]
        dist = np.sqrt(dx**2 + dy**2)
        if dist < 0.01:
            continue
        ux, uy = dx / dist, dy / dist
        # Start from factor edge, end at variable circle
        x0 = fpos[0] + ux * (sq / 2 + 0.05)
        y0 = fpos[1] + uy * (sq / 2 + 0.05)
        x1 = vpos[0] - ux * r
        y1 = vpos[1] - uy * r
        ax.plot([x0, x1], [y0, y1], color=COLORS['blue'], lw=2, zorder=3)

# Draw factor nodes (squares)
for fname, fpos in factor_nodes.items():
    draw_square_node(ax, fpos, fname.replace('$', ''),
                     size=sq, fc=COLORS['blue'], ec=COLORS['blue'],
                     fontsize=10, fontcolor='white')

# Draw variable nodes (circles)
for vname, vpos in var_nodes.items():
    draw_node(ax, vpos, vname, r, 'white', COLORS['blue'],
              fontsize=13, fontcolor=COLORS['blue'])

# Factorization annotation
ax.text(6, -1.2,
        r'$p(\mathbf{x}) = f_a(x_1,x_2)\, f_b(x_2,x_3)\, f_c(x_3,x_4)\,'
        r' f_d(x_4,x_5)\, f_e(x_2)\, f_f(x_4)$',
        fontsize=11, ha='center', va='center', color=COLORS['blue'])

# Legend
ax.plot(0, -1.8, 's', ms=12, color=COLORS['blue'], zorder=10)
ax.text(0.4, -1.8, 'Factor node', fontsize=10, va='center', color=COLORS['blue'])
c_legend = plt.Circle((3, -1.8), 0.18, fc='white', ec=COLORS['blue'], lw=1.5, zorder=10)
ax.add_patch(c_legend)
ax.text(3.4, -1.8, 'Variable node', fontsize=10, va='center', color=COLORS['blue'])

ax.set_title('Factor Graph', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig11_9_factor_graph')
plt.show()

## Figure 11.14 — Belief Propagation (Message Passing on a Factor Graph)

In the sum-product algorithm, messages are passed between variable nodes and factor nodes. Each message is a function of the receiving variable.

In [ ]:
# ── Figure 11.14: Belief Propagation (Message Passing) ─────────
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-1, 15)
ax.set_ylim(-2.5, 5)
ax.set_aspect('equal')
ax.axis('off')

r = 0.4        # variable node radius
sq = 0.5       # factor node size

# Chain factor graph: x1 — fa — x2 — fb — x3 — fc — x4
var_positions = {
    '$x_1$': (0, 1.5),
    '$x_2$': (4, 1.5),
    '$x_3$': (8, 1.5),
    '$x_4$': (12, 1.5),
}
fac_positions = {
    '$f_a$': (2, 1.5),
    '$f_b$': (6, 1.5),
    '$f_c$': (10, 1.5),
}

# Draw edges
chain_edges = [
    ('$x_1$', '$f_a$'), ('$f_a$', '$x_2$'),
    ('$x_2$', '$f_b$'), ('$f_b$', '$x_3$'),
    ('$x_3$', '$f_c$'), ('$f_c$', '$x_4$'),
]
for a, b in chain_edges:
    pos_a = var_positions.get(a, fac_positions.get(a))
    pos_b = var_positions.get(b, fac_positions.get(b))
    dx = pos_b[0] - pos_a[0]
    dy = pos_b[1] - pos_a[1]
    dist = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / dist, dy / dist
    # Determine start/end offsets
    r_a = r if a.startswith('$x') else sq / 2 + 0.05
    r_b = r if b.startswith('$x') else sq / 2 + 0.05
    x0 = pos_a[0] + ux * r_a
    y0 = pos_a[1] + uy * r_a
    x1 = pos_b[0] - ux * r_b
    y1 = pos_b[1] - uy * r_b
    ax.plot([x0, x1], [y0, y1], color='gray', lw=2.5, zorder=3)

# Draw factor nodes
for fname, fpos in fac_positions.items():
    draw_square_node(ax, fpos, fname.replace('$', ''),
                     size=sq, fc=COLORS['blue'], ec=COLORS['blue'],
                     fontsize=11, fontcolor='white')

# Draw variable nodes
for vname, vpos in var_positions.items():
    draw_node(ax, vpos, vname, r, 'white', COLORS['blue'],
              fontsize=14, fontcolor=COLORS['blue'])

# ── Draw message arrows ───────────────────────────────────────
# Forward messages (left to right) — shown above the chain
msg_fwd = [
    ((0, 1.5), (2, 1.5), r'$\mu_{x_1 \to f_a}$'),
    ((2, 1.5), (4, 1.5), r'$\mu_{f_a \to x_2}$'),
    ((4, 1.5), (6, 1.5), r'$\mu_{x_2 \to f_b}$'),
    ((6, 1.5), (8, 1.5), r'$\mu_{f_b \to x_3}$'),
    ((8, 1.5), (10, 1.5), r'$\mu_{x_3 \to f_c}$'),
    ((10, 1.5), (12, 1.5), r'$\mu_{f_c \to x_4}$'),
]

# Backward messages (right to left) — shown below
msg_bwd = [
    ((12, 1.5), (10, 1.5), r'$\mu_{x_4 \to f_c}$'),
    ((10, 1.5), (8, 1.5), r'$\mu_{f_c \to x_3}$'),
    ((8, 1.5), (6, 1.5), r'$\mu_{x_3 \to f_b}$'),
    ((6, 1.5), (4, 1.5), r'$\mu_{f_b \to x_2}$'),
    ((4, 1.5), (2, 1.5), r'$\mu_{x_2 \to f_a}$'),
    ((2, 1.5), (0, 1.5), r'$\mu_{f_a \to x_1}$'),
]

# Draw forward messages (above)
y_off_fwd = 0.9
for (x0, _), (x1, _), label in msg_fwd:
    mid_x = (x0 + x1) / 2
    ax.annotate('', xy=(x1 - 0.3, 1.5 + y_off_fwd),
                xytext=(x0 + 0.3, 1.5 + y_off_fwd),
                arrowprops=dict(arrowstyle='->', color=COLORS['red'],
                                lw=1.8, mutation_scale=14))
    ax.text(mid_x, 1.5 + y_off_fwd + 0.25, label, fontsize=8,
            ha='center', va='bottom', color=COLORS['red'])

# Draw backward messages (below)
y_off_bwd = -0.9
for (x0, _), (x1, _), label in msg_bwd:
    mid_x = (x0 + x1) / 2
    ax.annotate('', xy=(x1 + 0.3, 1.5 + y_off_bwd),
                xytext=(x0 - 0.3, 1.5 + y_off_bwd),
                arrowprops=dict(arrowstyle='->', color=COLORS['green'],
                                lw=1.8, mutation_scale=14))
    ax.text(mid_x, 1.5 + y_off_bwd - 0.35, label, fontsize=8,
            ha='center', va='top', color=COLORS['green'])

# Direction labels
ax.text(6, 3.8, 'Forward pass', fontsize=12, ha='center',
        color=COLORS['red'], fontweight='bold')
ax.annotate('', xy=(9, 3.5), xytext=(3, 3.5),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2.5))

ax.text(6, -1.5, 'Backward pass', fontsize=12, ha='center',
        color=COLORS['green'], fontweight='bold')
ax.annotate('', xy=(3, -1.2), xytext=(9, -1.2),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2.5))

# Marginal computation annotation
ax.text(6, -2.2,
        r'Marginal: $p(x_i) \propto \prod_{f \in \mathrm{ne}(x_i)}'
        r' \mu_{f \to x_i}(x_i)$',
        fontsize=12, ha='center', va='center', color=COLORS['blue'])

ax.set_title('Belief Propagation (Sum-Product Message Passing)', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig11_14_message_passing')
plt.show()

In [ ]:
print('All Chapter 11 figures generated.')